In [3]:
import numpy as np
import os
from tifffile import imread, imwrite
import glob


seg_2D_path_list = sorted(glob.glob("/media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose/output/*tif"))
seg_2D_path_list = sorted(
    seg_2D_path_list, 
    key=lambda x: int(os.path.basename(x).split('_z_')[-1].split('_')[0])
)

seg_2D_stack = []

for seg_2D_path in seg_2D_path_list:
    seg_2D = imread(seg_2D_path)
    seg_2D_stack.append(seg_2D)

seg_2D_stack = np.array(seg_2D_stack)
print(f"seg_2D_stack shape: {seg_2D_stack.shape}")


seg_2D_stack shape: (14, 3731, 3249)


In [4]:
import os
import numpy as np
from tifffile import imwrite
from segmentation_3D.match_2D_cells import matching_cells_2D
from multiprocessing import Pool
import matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries
from tqdm import tqdm

base_dir = "/media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose"
output_folder = os.path.join(base_dir, "output_Jimin_40X_dense_3D")

# -----------------------------
# Helper functions
# -----------------------------

def filter_single_slice_cells(seg_3D):
    """Remove cells that only appear in a single z-slice."""
    seg_3D_filtered = seg_3D.copy()
    labels = np.unique(seg_3D)
    labels = labels[labels != 0]  # skip background
    
    for label in labels:
        z_indices = np.where(np.any(seg_3D == label, axis=(1,2)))[0]
        if z_indices[-1] - z_indices[0] == 0:  # only one z slice
            seg_3D_filtered[seg_3D == label] = 0
    return seg_3D_filtered

from skimage.segmentation import find_boundaries
import numpy as np
import matplotlib.pyplot as plt

from skimage.morphology import binary_dilation, disk

import numpy as np
import matplotlib.pyplot as plt

def make_color_mask(seg_3D):
    """Assign each cell in seg_3D a random color from tab20 colormap."""
    labels = np.unique(seg_3D)
    labels = labels[labels != 0]  # skip background

    # Random color assignment from tab20
    cmap = plt.get_cmap("tab20")
    rng = np.random.default_rng()
    colors = (np.array([cmap(rng.integers(0, 20))[:3] for _ in labels]) * 255).astype(np.uint8)

    # Lookup table: index = label, value = RGB
    lut = np.zeros((seg_3D.max() + 1, 3), dtype=np.uint8)
    lut[labels] = colors

    # Vectorized mapping
    color_mask = lut[seg_3D]

    return color_mask



# -----------------------------
# Main processing
# -----------------------------

def process_JI_thre(JI_thre):
    print(f"Processing JI_thre: {JI_thre}")
    output_indexed = os.path.join(output_folder, f"seg_3D_JIthre_{JI_thre}.tif")
    if os.path.exists(output_indexed):
        print(f"Output for JI_thre {JI_thre} already exists. Skipping...")
        seg_3D = imread(output_indexed)
    else:
        seg_3D = matching_cells_2D(seg_2D_stack, JI_thre=JI_thre)

        # filter single-slice cells
        print("Before filtering, unique labels:", np.unique(seg_3D))
        seg_3D = filter_single_slice_cells(seg_3D)
        print("After filtering, unique labels:", np.unique(seg_3D))

        # save indexed mask
        print("Saving indexed mask...")
        imwrite(output_indexed, seg_3D.astype(np.uint8))

    # save color mask
    color_mask = make_color_mask(seg_3D)
    output_color = os.path.join(output_folder, f"seg_3D_JIthre_{JI_thre}_color.tif")
    imwrite(output_color, color_mask, photometric="rgb")

    return output_indexed, output_color


JI_thre_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
# JI_thre_list = [0.1]
with Pool() as pool:
    results = pool.map(process_JI_thre, JI_thre_list)

# results = []
# for JI_thre in JI_thre_list:
    # result = process_JI_thre(JI_thre)
    # results.append(result)

print("Processing completed. Output files:")
for result in results:
    print("Indexed:", result[0])
    print("Color:  ", result[1])


Processing JI_thre: 0.1Processing JI_thre: 0.2Processing JI_thre: 0.3Processing JI_thre: 0.4Processing JI_thre: 0.5Processing JI_thre: 0.6Processing JI_thre: 0.8Processing JI_thre: 0.7Processing JI_thre: 0.9










Matching slices: 100%|██████████| 13/13 [01:26<00:00,  6.62s/it]


Before filtering, unique labels: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 2

Matching slices: 100%|██████████| 13/13 [01:26<00:00,  6.68s/it]


Before filtering, unique labels: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 2

...

Before filtering, unique labels: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 2

Matching slices: 100%|██████████| 13/13 [01:28<00:00,  6.82s/it]


Before filtering, unique labels: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 2

Matching slices: 100%|██████████| 13/13 [01:30<00:00,  6.93s/it]


Before filtering, unique labels: [   0    1    2 ... 1024 1025 1026]
After filtering, unique labels: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  25  26  27  28  29  30  31  32  33  35  36  37
  38  39  40  41  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  61  62  63  64  67  68  69  70  72  73  74  75  76  77  78
  80  81  83  85  86  87  90  92  95 100 101 103 107 110 111 112 113 114
 116 118 119 121 124 128 132 133 134 135 137 138 141 143 146 147 149 150
 151 152 154 156 158 159 160 162 163 164 166 168 170 172 174 175 176 181
 182 183 189 191 193 194 195 197 198 199 200 201 202 204 206 207 208 209
 210 213 216 218 222 223 224 229 231 232 233 234 235 237 238 239 245 248
 249 252 253 256 257 258 259 260 261 264 265 266 267 269 270 271 272 279
 280 282 283 284 285 287 288 289 291 294 296 298 303 305 306 307 308 310
 311 313 314 316 318 319 320 321 322 325 326 329 330 332 334 336 338 339
 340 341 342 344 345 34

In [ ]:
import os
import numpy as np
from tifffile import imwrite
from segmentation_3D.match_2D_cells import matching_cells_2D
from multiprocessing import Pool
import matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries
from skimage.measure import regionprops

base_dir = "/media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose"
output_folder = os.path.join(base_dir, "output_Jimin_40X_dense_3D")

# -----------------------------
# Helper functions
# -----------------------------

def filter_single_slice_cells(seg_3D):
    """Remove cells that only appear in a single z-slice using regionprops."""
    seg_3D_filtered = seg_3D.copy()
    props = regionprops(seg_3D)

    for p in props:
        zmin, ymin, xmin, zmax, ymax, xmax = p.bbox
        if zmax - zmin == 1:  # only one slice in z
            seg_3D_filtered[seg_3D == p.label] = 0
    return seg_3D_filtered

def make_color_mask(seg_3D):
    """Generate an RGB color mask with random tab20 colors + 2px boundaries."""
    cmap = plt.get_cmap("tab20")
    labels = np.unique(seg_3D)
    labels = labels[labels != 0]  # skip background

    rng = np.random.default_rng()
    color_dict = {label: (np.array(cmap(rng.integers(0, 20)))[:3] * 255).astype(np.uint8)
                  for label in labels}

    color_mask = np.zeros(seg_3D.shape + (3,), dtype=np.uint8)

    for label in labels:
        color_mask[seg_3D == label] = color_dict[label]

        # draw 2-pixel boundaries
        boundaries = find_boundaries(seg_3D == label, connectivity=1, mode="outer")
        for _ in range(2):  # thicken to 2px
            boundaries = find_boundaries(boundaries, connectivity=1, mode="thick")
        color_mask[boundaries] = (0, 0, 0)  # black boundary

    return color_mask

# -----------------------------
# Main processing
# -----------------------------

def process_JI_thre(JI_thre):
    print(f"Processing JI_thre: {JI_thre}")
    seg_3D = matching_cells_2D(seg_2D_stack, JI_thre=JI_thre)

    # filter single-slice cells
    print("Before filtering, unique labels:", np.unique(seg_3D))
    seg_3D = filter_single_slice_cells(seg_3D)
    print("After filtering, unique labels:", np.unique(seg_3D))

    # save indexed mask
    output_indexed = os.path.join(output_folder, f"seg_3D_JIthre_{JI_thre}.tif")
    imwrite(output_indexed, seg_3D.astype(np.uint8))

    # save color mask
    print("Generating color mask...")
    color_mask = make_color_mask(seg_3D)
    output_color = os.path.join(output_folder, f"seg_3D_JIthre_{JI_thre}_color.tif")
    imwrite(output_color, color_mask, photometric="rgb")

    return output_indexed, output_color


JI_thre_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
with Pool() as pool:
    results = pool.map(process_JI_thre, JI_thre_list)

print("Processing completed. Output files:")
for result in results:
    print("Indexed:", result[0])
    print("Color:  ", result[1])


Processing JI_thre: 0.1Processing JI_thre: 0.5Processing JI_thre: 0.2Processing JI_thre: 0.4Processing JI_thre: 0.8Processing JI_thre: 0.3Processing JI_thre: 0.6Processing JI_thre: 0.7Processing JI_thre: 0.9










Matching slices:  31%|███       | 4/13 [00:27<01:00,  6.70s/it]Process ForkPoolWorker-338:
Process ForkPoolWorker-352:
Process ForkPoolWorker-381:
Process ForkPoolWorker-317:
Process ForkPoolWorker-296:
Process ForkPoolWorker-371:
Process ForkPoolWorker-304:
Process ForkPoolWorker-303:
Process ForkPoolWorker-343:
Process ForkPoolWorker-347:
Process ForkPoolWorker-383:
Process ForkPoolWorker-376:
Process ForkPoolWorker-368:
Process ForkPoolWorker-384:
Process ForkPoolWorker-373:
Process ForkPoolWorker-342:
Process ForkPoolWorker-322:
Process ForkPoolWorker-351:
Process ForkPoolWorker-294:
Process ForkPoolWorker-325:
Process ForkPoolWorker-363:
Process ForkPoolWorker-331:
Process ForkPoolWorker-299:
Process ForkPoolWorker-324:
Process ForkPoolWorker-361:
Process ForkPoolWorker-318:
Process ForkPoolWorker-356:
Process ForkPoolWorker-344:
Process ForkPoolWorker-350:
Process ForkPoolWorker-353:
Process ForkPoolWorker-360:
Process ForkPoolWorker-332:
Process ForkPoolWorker-377:
Process ForkP

In [ ]:
np.unique(seg_3D)

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18